# Colab bootstrap and project checks

Run this cell first when the notebook is opened directly from GitHub in Google Colab.

It clones the `week-12-execution-results` branch into the Colab runtime, updates an existing clone when possible, moves into the Week 12 project folder, and verifies that every required project data file is available before the rest of the notebook runs.

The OpenAI-compatible API key and endpoint are read only from **Google Colab Secrets**. They are never printed or written to the repository.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/mschemerii/UT-Austin.git"
REPO_BRANCH = "week-12-execution-results"
REPO_DIR = Path("/content/UT-Austin")
PROJECT_DIR = REPO_DIR / "week-12-final-project"

def run_git(args, cwd=None):
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stdout)
    return result.stdout.strip()

if not (REPO_DIR / ".git").exists():
    print(f"Cloning {REPO_BRANCH}...")
    result = subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stdout)
else:
    print("Existing repository clone found; refreshing branch...")
    run_git(["fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run_git(["checkout", REPO_BRANCH], cwd=REPO_DIR)
    # Update only when it can be fast-forwarded; do not overwrite local changes.
    pull = subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPO_BRANCH],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if pull.returncode != 0:
        print("Repository could not be fast-forwarded automatically.")
        print("Continuing with the current local checkout so local work is not overwritten.")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

REQUIRED_DATA_FILES = [
    "data/delivery_logs.csv",
    "data/ground_truth.csv",
    "data/customers.csv",
    "data/lockers.csv",
    "data/exception_resolution_playbook.txt",
]

missing = [p for p in REQUIRED_DATA_FILES if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Required project files are missing after cloning:\n- " + "\n- ".join(missing)
    )

print("\nProject data check:")
for p in REQUIRED_DATA_FILES:
    size = Path(p).stat().st_size
    print(f"  ✓ {p} ({size:,} bytes)")

print("\nRepository branch:", run_git(["branch", "--show-current"], cwd=REPO_DIR))
print("Repository commit:", run_git(["rev-parse", "--short", "HEAD"], cwd=REPO_DIR))
print("\nColab bootstrap complete.")


# Week 12 Final Project — AI-powered Last Mile Delivery Exception Handling Automation

**Full-code submission**

This notebook implements a proof-of-concept multi-agent system for delivery-exception triage, resolution, escalation, customer communication, and evaluation across the 10 curated shipments supplied with the project.


## Business Problem and Objective
Manual exception handling is slow and inconsistent because operations teams must interpret noisy delivery logs, customer context, locker availability, and policy under time pressure. The solution below combines deterministic preprocessing and escalation rules with specialized LLM agents and playbook retrieval.

The required evaluation dimensions are: task completion, escalation accuracy, tool-call accuracy, reasoning-trajectory coherence, and end-to-end latency. A task is complete only when exception classification, resolution action, and communication tone all match ground truth.


## Data
The project uses `delivery_logs.csv`, `ground_truth.csv`, `customers.db`, and `exception_resolution_playbook.pdf`. If the binary data files are not yet extracted, run `scripts/prepare_data.py` after cloning the repository.


In [ ]:
!pip -q install langchain-openai langgraph langchain-community langchain-huggingface langchain-chroma langchain-text-splitters pypdf sentence-transformers pandas numpy pydantic


In [ ]:
import os, re, json, time, sqlite3, subprocess, sys
from collections import defaultdict
from typing import TypedDict
import numpy as np, pandas as pd
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END
from IPython.display import display, Markdown

# Prepare exact course data from the checked-in encoded archive when needed.
if not os.path.exists('data/customers.db') and os.path.exists('scripts/prepare_data.py'):
    subprocess.run([sys.executable, 'scripts/prepare_data.py'], check=True)

def path(name):
    for p in (f'data/{name}', name):
        if os.path.exists(p): return p
    raise FileNotFoundError(name)

# Preferred: Google Colab Secrets. Nothing is printed or committed.
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

OPENAI_API_KEY=secret('OPENAI_API_KEY')
OPENAI_API_BASE=secret('OPENAI_API_BASE') or secret('OPENAI_BASE_URL')
if not OPENAI_API_KEY or not OPENAI_API_BASE:
    raise ValueError('Add OPENAI_API_KEY and OPENAI_API_BASE to Colab Secrets and enable notebook access.')

llm=ChatOpenAI(model='gpt-4o-mini',temperature=0,api_key=OPENAI_API_KEY,base_url=OPENAI_API_BASE,max_retries=2)
judge=ChatOpenAI(model='gpt-4o',temperature=0,api_key=OPENAI_API_KEY,base_url=OPENAI_API_BASE,max_retries=2)
print('Models configured from Colab Secrets.')
